##### Copyright 2026 Google LLC.

In [1]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini API: JSON 모드 빠른 시작

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/JSON_mode.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

Gemini API는 사용하려는 스키마를 설정하면 JSON 출력을 생성할 수 있습니다.

두 가지 방법을 사용할 수 있습니다. 원하는 출력 형식을 프롬프트에 직접 설정하거나, 스키마를 별도로 모델에 제공할 수 있습니다.

### 의존성 설치

In [ ]:
%pip install -U -q "google-genai>=1.0.0"

### API 키 설정

다음 셀을 실행하려면 API 키가 `GOOGLE_API_KEY`라는 이름의 Colab 시크릿에 저장되어 있어야 합니다. API 키가 없거나 Colab 시크릿 생성 방법을 모르는 경우, [인증 ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/Authentication.ipynb) 예제를 참고하세요.

In [3]:
from google.colab import userdata
from google import genai

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
client = genai.Client(api_key=GOOGLE_API_KEY)

## 프롬프트에서 제약된 출력 설정

첫 번째 예제에서는 프롬프트에서 원하는 스키마를 직접 설명합니다:


In [4]:
prompt = """
  List a few popular cookie recipes using this JSON schema:

  Recipe = {'recipe_name': str}
  Return: list[Recipe]
"""

이제 이 가이드에서 사용할 모델을 목록에서 선택하거나 직접 입력하세요. 2.5 모델과 같은 일부 모델은 thinking 모델이므로 응답 시간이 약간 더 걸립니다(자세한 내용은 [thinking 노트북](./Get_started_thinking.ipynb)을 참고하고, 특히 thinking을 끄는 방법을 확인하세요).

그런 다음 `config` 매개변수에서 `response_mime_type`을 지정하여 JSON 모드를 활성화합니다:

In [5]:
MODEL_ID = "gemini-3-flash-preview" # @param ["gemini-2.5-flash-lite", "gemini-2.5-flash", "gemini-2.5-pro", "gemini-2.5-flash-preview", "gemini-3.1-flash-lite-preview", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

raw_response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
    config={
        'response_mime_type': 'application/json'
    },
)

문자열을 JSON으로 파싱합니다:

In [6]:
import json

response = json.loads(raw_response.text)
print(response)

[{'recipe_name': 'Chocolate Chip Cookies'}, {'recipe_name': 'Oatmeal Raisin Cookies'}, {'recipe_name': 'Peanut Butter Cookies'}, {'recipe_name': 'Sugar Cookies'}, {'recipe_name': 'Snickerdoodles'}]


가독성을 위해 직렬화하여 출력합니다:

In [7]:
print(json.dumps(response, indent=4))

[
    {
        "recipe_name": "Chocolate Chip Cookies"
    },
    {
        "recipe_name": "Oatmeal Raisin Cookies"
    },
    {
        "recipe_name": "Peanut Butter Cookies"
    },
    {
        "recipe_name": "Sugar Cookies"
    },
    {
        "recipe_name": "Snickerdoodles"
    }
]


## 모델에 직접 스키마 제공

최신 모델(1.5 이상)에서는 스키마 객체(또는 동등한 파이썬 타입)를 직접 전달할 수 있으며, 출력은 해당 스키마를 엄격히 따릅니다.

이전 섹션과 동일한 예제를 사용하여 Recipe 타입을 다음과 같이 정의합니다:

In [8]:
import typing_extensions as typing

class Recipe(typing.TypedDict):
    recipe_name: str
    recipe_description: str
    recipe_ingredients: list[str]

`list[Recipe]` 객체 목록을 원하므로, `config`의 `response_schema` 필드에 `list[Recipe]`를 전달합니다.

In [9]:
result = client.models.generate_content(
    model=MODEL_ID,
    contents="List a few imaginative cookie recipes along with a one-sentence description as if you were a gourmet restaurant and their main ingredients",
    config={
        'response_mime_type': 'application/json',
        'response_schema': list[Recipe],
    },
)

In [10]:
print(json.dumps(json.loads(result.text), indent=4))

[
    {
        "recipe_name": "Lavender & White Chocolate Cloud Kisses",
        "recipe_description": "Delicate lavender-infused meringue cookies, airily light and meltingly tender, are kissed with swirls of creamy white chocolate.",
        "recipe_ingredients": [
            "Egg whites",
            "Sugar",
            "Dried lavender",
            "White chocolate"
        ]
    },
    {
        "recipe_name": "Smoked Paprika & Dark Chocolate Chili Sables",
        "recipe_description": "A sophisticated blend of smoky paprika, intense dark chocolate, and a whisper of chili creates a captivating sweet and savory experience in a crisp sable cookie.",
        "recipe_ingredients": [
            "All-purpose flour",
            "Unsalted butter",
            "Dark chocolate",
            "Smoked paprika",
            "Chili powder"
        ]
    },
    {
        "recipe_name": "Matcha & Black Sesame Shortbread Petals",
        "recipe_description": "Elegantly sculpted shortbread pet

이 방법은 호환되는 모델을 사용할 때 권장되는 방법입니다.

## 다음 단계
### 유용한 API 참고 자료:

[구조화된 출력](https://ai.google.dev/gemini-api/docs/structured-output) 문서 또는 [`GenerationConfig`](https://ai.google.dev/api/generate-content#generationconfig) API 참고 자료에서 자세한 내용을 확인하세요.

### 관련 예제

* 제약된 출력은 [텍스트 요약](../examples/json_capabilities/Text_Summarization.ipynb) 예제에서 모델에게 이야기를 요약하는 형식(장르, 등장인물 등)을 제공하는 데 사용됩니다.
* [객체 감지](../examples/Object_detection.ipynb) 예제는 JSON 제약 출력을 사용하여 감지 출력을 일관되게 만듭니다.

### Gemini API 탐색 계속하기

JSON이 모델 출력을 제한하는 유일한 방법은 아닙니다. [Enum](../quickstarts/Enum.ipynb)을 사용할 수도 있습니다. [함수 호출](../quickstarts/Function_calling.ipynb)과 [코드 실행](../quickstarts/Code_Execution.ipynb)은 자신의 함수를 사용하거나 모델이 직접 코드를 작성하고 실행하게 하는 방법으로 모델을 향상시키는 또 다른 방법입니다.